# 09 — Inpainting movie on a 10-second held-out burst

Take a **held-out validation burst file** (the trainer never saw it), apply the synthetic occlusion mask to every frame, run the U-Net, and show TRUE / MASKED / IMPUTED side-by-side as an animation with a running plasma-density panel.

This is the "does it actually work" figure: three direct comparisons (actual skymap, simulated-masked skymap, model-imputed skymap) plus a density metric over ~10 seconds of real burst data.

In [ ]:
import os, sys, warnings, numpy as np, matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
warnings.filterwarnings('ignore', category=DeprecationWarning); warnings.filterwarnings('ignore', category=FutureWarning)
ROOT = os.path.dirname(os.path.abspath('.')) if os.path.basename(os.getcwd())=='notebooks' else os.path.abspath('.')
sys.path.insert(0, os.path.join(ROOT, 'src')); sys.path.insert(0, os.path.join(ROOT, 'MMS-FPI-Data-Gaps'))
import data_pipeline as dp
from model import build_unet3d
from skymaps.skymap import Skymap
OUT = os.path.join(ROOT, 'outputs', 'unet_full_3rounds')

In [ ]:
mask = dp.synthetic_wedge_mask()
files = dp.find_dist_files(os.path.join(ROOT, 'MMS-FPI-Data-Gaps'))
_, val_files = dp.split_files(files, val_fraction=0.2, seed=0)
SUB = 4                                # 4*30ms = 120ms per frame
fb = dp.read_dist_file(val_files[0], subsample=SUB, with_phi=True)
X, Y = dp.build_inputs(fb, mask, use_pitch_angle=True, use_logb=True, temporal_window=1)
m = build_unet3d(base_filters=10, input_shape=(32, 16, 32, 5))
m.load_weights(os.path.join(OUT, 'model_latest.h5'))
P = m.predict(X, verbose=0)
y_cube = Y[..., 0]; p_cube = P[..., 0]
inpaint = y_cube.copy(); inpaint[:, mask] = p_cube[:, mask]
masked_in = X[..., 1]
n_keep = min(int(10*1000/(SUB*30)), fb.n_samples)
y_cube, masked_in, inpaint = y_cube[:n_keep], masked_in[:n_keep], inpaint[:n_keep]
print(f'kept {n_keep} frames = {n_keep*SUB*30/1000:.1f} s')

In [ ]:
def density(cube):
    sk = Skymap(cube.shape[0], name='x'); sk.skymap = np.zeros((cube.shape[0], 32, 16, 32), dtype=np.float64)
    sk.skymap[:] = dp.to_physical_space(cube); return np.asarray(sk.momsTS.density)
d_true = density(y_cube); d_recon = density(inpaint)
err = np.abs(d_true - d_recon) / np.maximum(np.abs(d_true), 1e-30)
print(f'median |dn|/n = {np.median(err):.3f}   mean = {err.mean():.3f}')

In [ ]:
E_BIN = 16
vmax = max(y_cube[:, E_BIN].max(), inpaint[:, E_BIN].max(), 1e-3)
real_time_s = SUB * 30 / 1000.0; t_axis = np.arange(n_keep)*real_time_s
fig = plt.figure(figsize=(11, 6.8))
gs = fig.add_gridspec(2, 3, height_ratios=[3.0, 1.6], hspace=0.45, wspace=0.10)
axT = fig.add_subplot(gs[0,0]); axM = fig.add_subplot(gs[0,1]); axR = fig.add_subplot(gs[0,2]); axD = fig.add_subplot(gs[1,:])
imT=axT.imshow(y_cube[0,E_BIN],vmin=0,vmax=vmax,origin='lower'); imM=axM.imshow(masked_in[0,E_BIN],vmin=0,vmax=vmax,origin='lower'); imR=axR.imshow(inpaint[0,E_BIN],vmin=0,vmax=vmax,origin='lower')
for ax,t in zip((axT,axM,axR),('TRUE','MASKED','U-Net IMPUTED')): ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
axD.plot(t_axis,d_true,'k-',lw=2.0,label='true'); axD.plot(t_axis,d_recon,'C0-',lw=1.4,label='U-Net imputed')
cursor=axD.axvline(0,color='red',lw=1.5); axD.set_xlabel('burst time (s)'); axD.set_ylabel('density (Skymap)'); axD.legend()
sup = fig.suptitle(f't = 0.00 s   fractional density error = {err[0]:.3f}')
def frame(i):
    imT.set_data(y_cube[i,E_BIN]); imM.set_data(masked_in[i,E_BIN]); imR.set_data(inpaint[i,E_BIN])
    cursor.set_xdata([t_axis[i], t_axis[i]]); sup.set_text(f't = {t_axis[i]:.2f} s   fractional density error = {err[i]:.3f}')
    return imT,imM,imR,cursor,sup
ani = FuncAnimation(fig, frame, frames=n_keep, blit=False, interval=int(real_time_s*1000))
gif_path = os.path.join(OUT, 'inpainting_movie.gif')
ani.save(gif_path, writer=PillowWriter(fps=int(round(1/real_time_s))))
plt.close(fig)
display(Image(gif_path))